**0. Imports**

In [ ]:
import pandas as pd
import numpy as np
import hashlib

**1. Extração (Extract)**

In [ ]:
# Leitura dos dados de origem com correção do CPF
# Fonte de dados omitida para preservação de informações sensíveis
df_2023 = pd.read_excel('resultado_2023.xlsx', dtype={'cpf_cnpj': str})
df_2024 = pd.read_excel('resultado_2024.xlsx', dtype={'cpf_cnpj': str})

In [ ]:
# Criação da coluna de ano do edital
df_2023["ano"] = 2023
df_2024["ano"] = 2024

In [ ]:
# Consolidação dos dados de múltiplas bases em uma única base
df = pd.concat([df_2023, df_2024], ignore_index=True)

In [ ]:
# Reorganiza para o 'ano' ser a primeira coluna
df = df[["ano"] + [col for col in df.columns if col != "ano"]]

**2. Transformação (Transform)**

A.   Tratamento LGPD e Criação de ID

In [ ]:
# Anonimização do CPF/CNPJ para proteger a privacidade dos alunos utilizando SHA-256
df["id_aluno"] = (
    df["cpf_cnpj"]
    .astype(str)
    .apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
)

B. Datas e Idade

In [ ]:
# Conversão das datas de nascimento e ingresso para o formato datetime
df["dt_nascimento"] = pd.to_datetime(df["dt_nascimento"], format="%d/%m/%Y")
df["dt_ingresso"] = pd.to_datetime(df["dt_ingresso"], format="%d/%m/%Y")
df["idade_ingresso"] = (
    (df["dt_ingresso"] - df["dt_nascimento"]).dt.days // 365.25
).astype(int)

df["UF_Mapa"] = df["uf_residencial"] + ", Brazil"

df["faixa_etaria"] = pd.cut(
    df["idade_ingresso"],
    bins=[0, 19, 24, 29, 34, 39, 200],
    labels=[
        "Menos de 20",
        "20-24",
        "25-29",
        "30-34",
        "35-39",
        "40+"
    ]
)

C.   Limpeza e padronização

In [ ]:
# Remoção de colunas pessoais, acadêmicas e administrativas
df = df.drop(columns=[
    "codigo",
    "nome",
    "matricula",
    "turma_entrada",
    "situacao_trabalho_final",
    "orientador",
    "orientacao",
    "tipo_orientacao",
    "data_defesa",
    "cpf_cnpj",
    "dt_nascimento",
    "curso",
    "modalidade",
    "email_aluno",
    "email_alternativo",
    "email_usuario",
    "telefone_fixo",
    "telefone_celular",
    "passaporte",
    "municipio_residencial",
    "edital"]
)

df["polo"] = df["polo"].str.split(" - ").str[1]
df["raca"] = df["raca"].replace("Preto", "Preta")
df["tipo_da_necessidade"] = df["tipo_da_necessidade"].replace("Deficiência Visual - Cegueira", "Deficiência Visual")
df["data_ocorrencia"] = df["data_ocorrencia"].str.replace(
    r'/0(\d{2})$', r'/20\1', regex=True
)
df["data_ocorrencia"] = pd.to_datetime(
    df["data_ocorrencia"],
    format="%d/%m/%Y",
    errors="coerce"
)

D. Regras de Negócio (Reingresso e Carga Horária)

In [ ]:
# Identifica a matrícula mais recente para tratamento de reingressos
df = df.sort_values(by=['id_aluno', 'ano'])
df['matricula_mais_recente'] = ~df.duplicated(subset=['id_aluno'], keep='last')

In [ ]:
# Cálculo da Carga Horária e Conclusão
carga_por_ano = {
    2023: 510,
    2024: 540
}
df["carga_horaria_exigida"] = df["ano"].map(carga_por_ano)
df["percentual_conclusao"] = (df["carga_horaria_feita"] / df["carga_horaria_exigida"]).round(2)

**3. Engenharia de Atributos Analíticos**

In [ ]:
# Perfil de Evasão
df["perfil_evasao"] = np.where(
    df["situacao_discente"] == "CANCELADO",
    np.select(
        [
            df["percentual_conclusao"].isna(),
            df["percentual_conclusao"] <= 0.295
        ],
        [
            "Imediata",
            "Precoce"
        ],
        default="Tardia"
    ),
    None
)

In [ ]:
# Tempo até evasão
df["meses_ate_evasao"] = (
    (df["data_ocorrencia"] - df["dt_ingresso"]).dt.days / 30.44
).round(1)

df.loc[
    df["situacao_discente"] != "CANCELADO",
    "meses_ate_evasao"
] = np.nan

df.loc[
    df["meses_ate_evasao"] < 0,
    "meses_ate_evasao"
] = np.nan

In [ ]:
# Ajuste da data para o PowerBI
df["dt_ingresso"] = df["dt_ingresso"].dt.date
df["data_ocorrencia"] = df["data_ocorrencia"].dt.date

**4. Carregamento (Load)**

In [ ]:
# Base final preparada para consumo no Power BI
df.to_csv(
    "base_geo_completa.csv", 
    index=False, 
    encoding="utf-8-sig", 
    sep=";", 
    decimal=","
)